In [1]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
# Load a CSV file into a DataFrame


In [2]:
df = pd.read_csv('IMDB-Movie-Dataset(2023-1951).csv')

print(df.head())

   Unnamed: 0    movie_id                       movie_name  year  \
0           0  tt15354916                            Jawan  2023   
1           1  tt15748830                       Jaane Jaan  2023   
2           2  tt11663228                           Jailer  2023   
3           3  tt14993250  Rocky Aur Rani Kii Prem Kahaani  2023   
4           4  tt15732324                            OMG 2  2023   

                   genre                                           overview  \
0       Action, Thriller  A high-octane action thriller which outlines t...   
1  Crime, Drama, Mystery  A single mother and her daughter who commit a ...   
2  Action, Comedy, Crime  A retired jailer goes on a manhunt to find his...   
3  Comedy, Drama, Family  Flamboyant Punjabi Rocky and intellectual Beng...   
4          Comedy, Drama  An unhappy civilian asks the court to mandate ...   

            director                                               cast  
0              Atlee  Shah Rukh Khan, Naya

In [3]:
# List all column names
print("Columns in the DataFrame:")
print(df.columns.tolist())


Columns in the DataFrame:
['Unnamed: 0', 'movie_id', 'movie_name', 'year', 'genre', 'overview', 'director', 'cast']


In [4]:
# To See Unique genres
# Split genres into lists and clean up whitespace
df['genre_list'] = df['genre'].str.split(',').apply(lambda x: [genre.strip() for genre in x])

# Explode the genre lists into separate rows
genres_exploded = df.explode('genre_list')

# Get unique genres
unique_genres = genres_exploded['genre_list'].unique()

# Count total unique genres
total_unique_genres = len(unique_genres)
print("Total unique genres:", total_unique_genres)

# Display unique genres
print("\nList of unique genres:")
for genre in unique_genres:
    print(genre)

Total unique genres: 20

List of unique genres:
Action
Thriller
Crime
Drama
Mystery
Comedy
Family
Adventure
Fantasy
History
Musical
Biography
Horror
Romance
Sport
Sci-Fi

Music
Animation
War


In [5]:
df['genre'] = df['genre'].str.strip()
missing_genre_rows = df[df['genre'].isnull() | (df['genre'] == '')]
print("Missing values in the genre column  : " , len(missing_genre_rows))

df = df.dropna(subset=['genre'])  # Remove rows with NaN values in 'genre' column
df = df[df['genre'] != '']  # Remove rows where 'genre' is an empty string
 # Check the number of remaining rows

Missing values in the genre column  :  18


In [6]:
df['genre'] = df['genre'].str.strip()
missing_genre_rows = df[df['genre'].isnull() | (df['genre'] == '')]
print(len(missing_genre_rows))

0


In [7]:
# To See Unique genres
# Split genres into lists and clean up whitespace
df['genre_list'] = df['genre'].str.split(',').apply(lambda x: [genre.strip() for genre in x])

# Explode the genre lists into separate rows
genres_exploded = df.explode('genre_list')

# Get unique genres
unique_genres = genres_exploded['genre_list'].unique()

# Count total unique genres
total_unique_genres = len(unique_genres)
print("Total unique genres:", total_unique_genres)

# Display unique genres
print("\nList of unique genres:")
for genre in unique_genres:
    print(genre)

Total unique genres: 19

List of unique genres:
Action
Thriller
Crime
Drama
Mystery
Comedy
Family
Adventure
Fantasy
History
Musical
Biography
Horror
Romance
Sport
Sci-Fi
Music
Animation
War


In [8]:
# Count the number of rows where the 'overview' column is either "Add a Plot" or "Plot under wraps"
add_plot_count = df[df['overview'] == 'Add a Plot'].shape[0]
plot_under_wraps_count = df[df['overview'] == 'Plot under wraps'].shape[0]

print(f"Number of 'Add a Plot' entries in 'overview' column: {add_plot_count}")
print(f"Number of 'Plot under wraps' entries in 'overview' column: {plot_under_wraps_count}")


Number of 'Add a Plot' entries in 'overview' column: 65
Number of 'Plot under wraps' entries in 'overview' column: 1


In [9]:
# Remove rows where the 'overview' column is "Add a Plot" or "Plot under wraps"
df = df[~df['overview'].isin(['Add a Plot', 'Plot under wraps'])]

# Verify the removal
print(f"Remaining rows after removal: {len(df)}")


Remaining rows after removal: 2115


In [10]:
# Count the number of rows where the 'overview' column is either "Add a Plot" or "Plot under wraps"
add_plot_count = df[df['overview'] == 'Add a Plot'].shape[0]
plot_under_wraps_count = df[df['overview'] == 'Plot under wraps'].shape[0]

print(f"Number of 'Add a Plot' entries in 'overview' column: {add_plot_count}")
print(f"Number of 'Plot under wraps' entries in 'overview' column: {plot_under_wraps_count}")

Number of 'Add a Plot' entries in 'overview' column: 0
Number of 'Plot under wraps' entries in 'overview' column: 0


In [11]:
df = df.dropna().reset_index(drop=True)

In [12]:
# Load the pre-trained BERT tokenizer and model with explicit clean_up_tokenization_spaces setting
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', clean_up_tokenization_spaces=True)
model = BertModel.from_pretrained('bert-base-uncased')

In [13]:
def generate_movie_embedding(description):
    # Tokenize the movie description
    inputs = tokenizer(description, return_tensors="pt", padding=True, truncation=True, max_length=512)
    
    # Get the embeddings from BERT
    outputs = model(**inputs)
    
    # Average the token embeddings to get a single movie embedding (you can also use other methods)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Average across tokens
    return embeddings

In [14]:
unique_genres = df['genre'].unique()

# Create a dictionary that maps each genre to an index
genre_to_index = {genre: idx for idx, genre in enumerate(unique_genres)}

# Step 2: One-hot Encoding for Genres
def generate_genre_embedding(genres, genre_to_index):
    # Create a zero vector of size equal to the number of unique genres
    genre_vector = np.zeros(len(unique_genres))  
    
    # Set the corresponding index to 1 for each genre
    for genre in genres:
        genre = genre.strip()  # Remove any extra spaces from the genre
        genre_index = genre_to_index.get(genre)  # Get the index for the genre
        if genre_index is not None:
            genre_vector[genre_index] = 1
    
    # Reshape the genre_vector to ensure it's a 2D array
    genre_embedding = genre_vector.reshape(1, -1)
    
    return genre_embedding

In [15]:
# Step 1: Compute Overview Embeddings
overview_embeddings = []

for i, row in df.iterrows():
    description = row['overview']  # Assuming 'overview' contains the movie description
    
    # Get the movie description embedding
    embedding = generate_movie_embedding(description)
    embedding = embedding.detach().numpy().reshape(1, -1)  # Ensure it's 2D
    
    overview_embeddings.append(embedding)

# Convert the list of overview embeddings into a NumPy array
overview_embeddings = np.vstack(overview_embeddings)

# Step 2: Apply PCA to Overview Embeddings
# Standardize the overview embeddings before PCA
scaler_overview = StandardScaler()
overview_embeddings_standardized = scaler_overview.fit_transform(overview_embeddings)

# Apply PCA to reduce dimensionality of overview embeddings
pca_overview = PCA(n_components=300)  # Retain 100 components or adjust as needed
reduced_overview_embeddings = pca_overview.fit_transform(overview_embeddings_standardized)

# Verify the shape and explained variance
print("Original shape of overview embeddings:", overview_embeddings.shape)
print("Reduced shape of overview embeddings:", reduced_overview_embeddings.shape)
print("Explained Variance (Overview, first 10 components):", pca_overview.explained_variance_ratio_[:10])
print("Total Variance Retained (Overview):", np.sum(pca_overview.explained_variance_ratio_))

# Step 3: Generate Genre One-Hot Encodings
genre_embeddings = []

for i, row in df.iterrows():
    # Get genres as a list (e.g., ['Action', 'Comedy', 'Drama'])
    genres = row['genre'].split(",")
    
    # Generate one-hot encoding for the genres
    genre_embedding = generate_genre_embedding(genres, genre_to_index)
    genre_embeddings.append(genre_embedding)


# Convert genre embeddings to a NumPy array
genre_embeddings = np.array(genre_embeddings)



# Reshape genre_embeddings to ensure it's 2D (if needed)
if genre_embeddings.ndim == 3:
    genre_embeddings = genre_embeddings.squeeze(axis=1)

# Step 4: Combine Reduced Overview Embeddings and Genre One-Hot Encodings
combined_embeddings = np.hstack((reduced_overview_embeddings, genre_embeddings))

# Verify the shape of the final movie embeddings
print("Final shape of combined movie embeddings:", combined_embeddings.shape)




Original shape of overview embeddings: (2078, 768)
Reduced shape of overview embeddings: (2078, 300)
Explained Variance (Overview, first 10 components): [0.09182159 0.06607978 0.0490037  0.03544463 0.02943625 0.02560963
 0.02455486 0.02364872 0.02141056 0.01887356]
Total Variance Retained (Overview): 0.95188034
Final shape of combined movie embeddings: (2078, 517)


In [16]:
df.head()

,Unnamed: 0,movie_id,movie_name,year,genre,overview,director,cast,genre_list
0,0,tt15354916,Jawan,2023,"Action, Thriller",A high-octane action thriller which outlines t...,Atlee,"Shah Rukh Khan, Nayanthara, Vijay Sethupathi, ...","[Action, Thriller]"
1,1,tt15748830,Jaane Jaan,2023,"Crime, Drama, Mystery",A single mother and her daughter who commit a ...,Sujoy Ghosh,"Kareena Kapoor, Jaideep Ahlawat, Vijay Varma, ...","[Crime, Drama, Mystery]"
2,2,tt11663228,Jailer,2023,"Action, Comedy, Crime",A retired jailer goes on a manhunt to find his...,Nelson Dilipkumar,"Rajinikanth, Mohanlal, Shivarajkumar, Jackie S...","[Action, Comedy, Crime]"
3,3,tt14993250,Rocky Aur Rani Kii Prem Kahaani,2023,"Comedy, Drama, Family",Flamboyant Punjabi Rocky and intellectual Beng...,Karan Johar,"Ranveer Singh, Alia Bhatt, Dharmendra, Shabana...","[Comedy, Drama, Family]"
4,4,tt15732324,OMG 2,2023,"Comedy, Drama",An unhappy civilian asks the court to mandate ...,Amit Rai,"Pankaj Tripathi, Akshay Kumar, Yami Gautam, Pa...","[Comedy, Drama]"


In [17]:
# Step 5: Add Embeddings to DataFrame
df['overview_embedding'] = list(reduced_overview_embeddings)  # Add reduced overview embeddings
df['genre_embedding'] = list(genre_embeddings)  # Add genre one-hot encodings
df['combined_embedding'] = list(combined_embeddings)  # Add combined embeddings



In [18]:
# Display the updated DataFrame
df.head()

,Unnamed: 0,movie_id,movie_name,year,genre,overview,director,cast,genre_list,overview_embedding,genre_embedding,combined_embedding
0,0,tt15354916,Jawan,2023,"Action, Thriller",A high-octane action thriller which outlines t...,Atlee,"Shah Rukh Khan, Nayanthara, Vijay Sethupathi, ...","[Action, Thriller]","[-5.0708923, 9.435328, 0.2523424, -12.033529, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-5.070892333984375, 9.435327529907227, 0.2523..."
1,1,tt15748830,Jaane Jaan,2023,"Crime, Drama, Mystery",A single mother and her daughter who commit a ...,Sujoy Ghosh,"Kareena Kapoor, Jaideep Ahlawat, Vijay Varma, ...","[Crime, Drama, Mystery]","[-11.981152, 0.09720064, 1.3757641, 4.408209, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...","[-11.981151580810547, 0.09720063954591751, 1.3..."
2,2,tt11663228,Jailer,2023,"Action, Comedy, Crime",A retired jailer goes on a manhunt to find his...,Nelson Dilipkumar,"Rajinikanth, Mohanlal, Shivarajkumar, Jackie S...","[Action, Comedy, Crime]","[-6.290186, 4.291925, -4.3512173, 1.4714161, -...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-6.290185928344727, 4.291924953460693, -4.351..."
3,3,tt14993250,Rocky Aur Rani Kii Prem Kahaani,2023,"Comedy, Drama, Family",Flamboyant Punjabi Rocky and intellectual Beng...,Karan Johar,"Ranveer Singh, Alia Bhatt, Dharmendra, Shabana...","[Comedy, Drama, Family]","[4.2022715, -5.1491327, 9.279877, -3.117734, 3...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, ...","[4.202271461486816, -5.14913272857666, 9.27987..."
4,4,tt15732324,OMG 2,2023,"Comedy, Drama",An unhappy civilian asks the court to mandate ...,Amit Rai,"Pankaj Tripathi, Akshay Kumar, Yami Gautam, Pa...","[Comedy, Drama]","[-8.410875, 4.378258, -3.6146255, 3.7010746, -...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...","[-8.41087532043457, 4.378258228302002, -3.6146..."


In [19]:
movie_embeddings = combined_embeddings

In [20]:
# Example user profile: User liked 'Godfather' and 'K.G.F: Chapter 1'
#input = ['Rockstar','Tumbbad']
input = ['Krrish']
#input =['Sukhee']

# Find the indices of the liked movies in the dataframe
user_movie_indices = [df[df['movie_name'] == movie].index[0] for movie in input]

# Calculate user profile by averaging the embeddings of the liked movies
user_profile = np.mean([movie_embeddings[i] for i in user_movie_indices], axis=0)

# Print the liked movie names and the user profile
print("input Movies:", input)
print("User Profile as a list:", user_profile.tolist())

user_movie_indices


input Movies: ['Krrish']
User Profile as a list: [2.8262195587158203, 2.937276840209961, -9.10635757446289, -1.2455977201461792, 7.160590648651123, -8.42646598815918, -9.456833839416504, 3.7645955085754395, -9.275101661682129, -7.332108974456787, 0.43302348256111145, -10.702631950378418, -1.5046932697296143, 3.210906505584717, 6.539816856384277, 2.7931292057037354, 2.8133156299591064, -0.8300255537033081, 3.5512566566467285, 7.1526641845703125, 3.2801318168640137, -1.5632977485656738, 4.328505992889404, 1.8529689311981201, -3.3250820636749268, -2.985689878463745, 0.8822378516197205, -1.874825358390808, -1.5486485958099365, -3.8359556198120117, 0.6635167002677917, 2.5353376865386963, -5.196525573730469, -2.462226152420044, -1.5068769454956055, 3.099778413772583, 2.3372700214385986, 0.7185608148574829, -1.9570485353469849, -2.7479748725891113, -2.0133917331695557, -0.12539134919643402, 2.0083160400390625, 2.426643133163452, -0.009363860823214054, 1.0363439321517944, -1.8689574003219604, 

[296]

In [21]:
liked_movies = df['movie_name'][user_movie_indices].tolist()

# Print the movie names
print("Movies given:", liked_movies)

Movies given: ['Krrish']


In [22]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

k = 10+len(input)  # Number of nearest neighbors you want to find
knn = NearestNeighbors(n_neighbors=k, metric='cosine')  # Use cosine distance for similarity

# Step 2: Fit the model with all movie embeddings (no filtering yet)
knn.fit(movie_embeddings)

# Step 3: Find the k nearest neighbors to the user profile
distances, indices = knn.kneighbors(user_profile.reshape(1, -1), n_neighbors=k)

# Step 4: Get the recommended movie names using the indices from the original movie embeddings
recommended_movies = df.iloc[indices.flatten()]['movie_name']

# Step 5: Remove the liked movies from the recommendations
filtered_recommendations = [movie for movie in recommended_movies if movie not in liked_movies]

# Step 6: Display the top recommended movies (after filtering out the liked ones)
print("Recommended Movies:", filtered_recommendations)

Recommended Movies: ['A Flying Jatt', 'Bhavesh Joshi Superhero', 'Brahmastra Part One: Shiva', 'Mr. X', 'Om - The Battle Within', 'Krrish 3', 'Commando 2', 'Street Fighter: The Legend of Chun-Li', 'Ganapath', 'Kalki 2898-AD']


In [23]:
# List of user liked movies
user_liked_movies = liked_movies

# Initialize an empty list to store the genres of user liked movies
user_movie_genres = []

# Loop through each movie and fetch its genre(s)
for movie in user_liked_movies:
    # Find the genre(s) for the movie from the df DataFrame
    genre = df[df['movie_name'] == movie]['genre'].values
    if len(genre) > 0:
        user_movie_genres.append((movie, genre[0]))  # Get the first genre if available
    else:
        user_movie_genres.append((movie, 'Genre Not Found'))  # If no genre found

# Print the genres for each user liked movie
for movie, genre in user_movie_genres:
    print(f"Movie: {movie}, Genre: {genre}")


Movie: Krrish, Genre: Action, Adventure, Sci-Fi


In [24]:
recommended_genres = []

# Loop through each recommended movie and fetch its genres
for movie in filtered_recommendations:
    # Find the genre(s) for the movie from the df DataFrame
    genre = df[df['movie_name'] == movie]['genre'].values
    if len(genre) > 0:
        recommended_genres.append((movie, genre[0]))  # Get the first genre if available
    else:
        recommended_genres.append((movie, 'Genre Not Found'))  # If no genre found

# Print the genres for each recommended movie
for movie, genre in recommended_genres:
    print(f"Movie: {movie}, Genre: {genre}")

Movie: A Flying Jatt, Genre: Action, Adventure, Comedy
Movie: Bhavesh Joshi Superhero, Genre: Action, Crime, Drama
Movie: Brahmastra Part One: Shiva, Genre: Action, Adventure, Fantasy
Movie: Mr. X, Genre: Action, Crime, Drama
Movie: Om - The Battle Within, Genre: Action, Thriller
Movie: Krrish 3, Genre: Action, Adventure, Sci-Fi
Movie: Commando 2, Genre: Action, Adventure, Thriller
Movie: Street Fighter: The Legend of Chun-Li, Genre: Action, Crime, Fantasy
Movie: Ganapath, Genre: Action, Drama, Sci-Fi
Movie: Kalki 2898-AD, Genre: Action, Drama, Fantasy


In [25]:
recommended_genres_overview = []

# Loop through each recommended movie and fetch its genre and overview
for movie in filtered_recommendations:
    # Find the genre and overview for the movie from the df DataFrame
    movie_data = df[df['movie_name'] == movie]
    if not movie_data.empty:
        genre = movie_data['genre'].values[0]  # Get the genre
        overview = movie_data['overview'].values[0]  # Get the overview
        recommended_genres_overview.append((movie, genre, overview))
    else:
        recommended_genres_overview.append((movie, 'Genre Not Found', 'Overview Not Found'))  # If no data found

# Print the genre and overview for each recommended movie
for movie, genre, overview in recommended_genres_overview:
    print(f"Movie: {movie}, Genre: {genre},--- Overview: {overview}")


Movie: A Flying Jatt, Genre: Action, Adventure, Comedy,--- Overview: Jatt is a reluctant superhero who fights crime and protects people. He meets his match in the evil Raka, who he must vanquish to save the day.
Movie: Bhavesh Joshi Superhero, Genre: Action, Crime, Drama,--- Overview: The origin story of Bhavesh Joshi, an Indian superhero, who sets out to fulfill his slain friend's wish to clean and reform the country, by training himself to fight and wearing a mask.
Movie: Brahmastra Part One: Shiva, Genre: Action, Adventure, Fantasy,--- Overview: A DJ with superpowers and his ladylove embark on a mission to protect the Brahmastra, a weapon of enormous energy, from dark forces closing in on them.
Movie: Mr. X, Genre: Action, Crime, Drama,--- Overview: After gaining the power of invisibility; a man becomes a vigilante, in order to take revenge on those who have wronged him.
Movie: Om - The Battle Within, Genre: Action, Thriller,--- Overview: After losing his memory while fighting enemi